# <font face="Verdana" size=6 color='#6495ED'>  IAD 011 - APRENDIZADO POR REFORÇO

<font face="Verdana" size=3 color='#40E0D0'> Profs. Larissa Driemeier e Thiago Martins


<center><img src='https://drive.google.com/uc?export=view&id=1PoKxE7mE-pgBYahdjXJcfC1axNkIAYS9' width="600"></center>

Material para aula sobre Model Free do Curso [IAD-011 Aprendizado por reforço](https://alunoweb.net/moodle/pluginfile.php/157552/mod_resource/content/1/RL_T05-Y2025.pdf)

__Livro texto:__

Richard S. Sutton and Andrew G. Barto, [Reinforcement learning : an introduction](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf), The MIT Press, 2nd ed., 2018.
__Referências:__

Para compor este Notebook, foram usadas as seguintes referências, além do livro texto:
1. [Windy Grid World Solution(using SARSA_Q-learning)](https://www.kaggle.com/code/kamal007/windy-grid-world-solution-using-sarsa-q-learning/notebook)

In [ ]:
import numpy as np
import pandas as pd
import random

import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = "serif"
import copy

import ipywidgets as widgets
from IPython.display import display

import urllib.request
import json

from sklearn.metrics import mean_squared_error

from collections import defaultdict

import time


# Monte Carlo

## Random Walk com 5 estados

Vamos implementar TD(0) para estimar $v_\pi$ do tradicional problema *Random Walk*.

<center><img src='https://drive.google.com/uc?export=view&id=1I5jIH-k-6Pc7dFDA3LF9ZZXiLu1Uvtvk' width="900"></center>



* Todos os episódios começam no estado central 𝐶;
* O agente vai para a esquerda ou para a direita, com igual probabilidade;
* Os episódios terminam na extrema esquerda (festa) ou na extrema direita (aula);
* Quando um episódio termina à direita, na aula, ocorre uma recompensa de 1; todos outras recompensas são zero;
* Como esta tarefa não é descontada e é episódica, o verdadeiro valor de cada estado é a probabilidade de terminar à direita se começar daquele estado: os valores verdadeiros de todos os estados, de 𝐴 a 𝐸, são $v_\pi = \begin{bmatrix}1/6 & 2/6 & 3/6 & 4/6 & 5/6\end{bmatrix}$.



Ou seja, a resposta final do exercício deve ser:
`ValoresEstados[1:-1] = [ 0.1667,  0.3333,  0.5   ,  0.6667,  0.8333]`


In [ ]:
class RandomWalk_MCM():
    def __init__(self, alpha = None, gamma = 1.0, valores = np.ones(7)*0.5):
        self.estados = ["Festa", "A", "B", "C", "D", "E", "Aula"]      # Todos os estados
        self.posicao = 3                                               # Posição inicial é "C"
        self.estado = self.estados[self.posicao]
        self.valores = valores
        self.end = False
        self.alpha = alpha
        self.gamma = gamma      # Fator de desconto para recompensas futuras (=1 possivel porque o jogo é finito)

    def escolhe_acao(self):
        if not self.end:
            acao = np.random.choice(["esquerda", "direita"])

            if acao == "esquerda":
                #Movimento para Esquerda
                nova_posicao = self.posicao - 1
            elif acao == "direita":
                #Movimento para Direita
                nova_posicao = self.posicao + 1

            recompensa = self.get_recompensa(nova_posicao)
            self.posicao = nova_posicao
            self.estado = self.estados[self.posicao]
            if (self.estado == "Festa") or (self.estado == "Aula"):
                #Random Walk acabou!
                self.end = True
        else:
            print("A movimentação não é possível porque o Random Walk já terminou.")
        return recompensa

    def get_recompensa(self, nova_posicao):
        return 1.0 if self.estados[nova_posicao] == "Aula" else 0.0

    def novo_episodio(self):
        self.posicao = 3
        self.estado = self.estados[self.posicao]
        self.end = False

    def MCM_FirstVisit(self, Contagem_visitas, gamma):
        self.novo_episodio()
        Episodio_estados = []
        Episodio_recompensas = []
        Episodio_estados.append(self.estado)

        while not self.end:
            recompensa = self.escolhe_acao()
            Episodio_estados.append(self.estado)
            Episodio_recompensas.append(recompensa)

        G = 0
        visitados = set()
        for estado, recompensa in zip(reversed(Episodio_estados), reversed(Episodio_recompensas)):
            G = recompensa + self.gamma * G
            if estado not in visitados:
                s = self.estados.index(estado)
                n = Contagem_visitas.get(s, 0) + 1
                a = self.alpha if self.alpha else 1/n
                delta = G - self.valores[s]
                self.valores[s] += a * delta
                Contagem_visitas[s] = n
                visitados.add(estado)
        return self.valores, Contagem_visitas, Episodio_estados

    def MCM_EveryVisit(self, Contagem_visitas, gamma):
        self.novo_episodio()
        Episodio_estados = []
        Episodio_recompensas = []
        Episodio_estados.append(self.estado)

        while not self.end:
            recompensa = self.escolhe_acao()
            Episodio_estados.append(self.estado)
            Episodio_recompensas.append(recompensa)

        G = 0
        for estado, recompensa in zip(reversed(Episodio_estados), reversed(Episodio_recompensas)):
            G = recompensa + self.gamma * G
            s = self.estados.index(estado)
            n = Contagem_visitas.get(s, 0) + 1
            a = self.alpha if self.alpha else 1/n
            delta = G - self.valores[s]
            self.valores[s] += a * delta
            Contagem_visitas[s] = n
        return self.valores, Contagem_visitas, Episodio_estados


In [ ]:
def RandomWalk_avalia_v_MCM(Mike, max_episodios, gamma, ValoresEstadosTabela, EpisodioEstadosTabela, epocasPlot, metodo='FirstVisit'):
    Contagem_visitas = {}  # Dicionário para armazenar total de vezes que o estado foi visitado
    for epoca in range(max_episodios):
        if metodo == 'FirstVisit':
            ValoresEstados, Contagem_visitas, Episodios_estados = Mike.MCM_FirstVisit(Contagem_visitas, gamma)
        elif metodo == 'EveryVisit':
            ValoresEstados, Contagem_visitas, Episodios_estados = Mike.MCM_EveryVisit(Contagem_visitas, gamma)
        else:
            raise ValueError(f'Método "{metodo}" não reconhecido. Use "FirstVisit" ou "EveryVisit".')

        if epoca in epocasPlot:
            ValoresEstadosTabela.append(np.array(ValoresEstados[1:-1]))
            EpisodioEstadosTabela.append(Episodios_estados)
    return ValoresEstadosTabela, EpisodioEstadosTabela


In [ ]:
ValoresEstados = np.ones(7) * 0.5
ValoresEstados[0] = 0
ValoresEstados[-1] = 0
max_episodios  = 101
gamma = 1.0
epocasPlot = [0,1, 10,  100]
ValoresEstadosTabela = []
EpisodioEstadosTabela = []
Mike = RandomWalk_MCM(valores = ValoresEstados)
ValoresEstadosTabela,EpisodioEstadosTabela = RandomWalk_avalia_v_MCM(Mike, max_episodios, gamma, ValoresEstadosTabela,
                                                                     EpisodioEstadosTabela, epocasPlot)#,metodo='EveryVisit')
print(ValoresEstadosTabela)
print(EpisodioEstadosTabela)


In [ ]:
def plot_Valores():
  x = ['A','B', 'C','D', 'E']
  epocasPlot = [0,1, 10,  100]
  ValoresIniciais = np.ones(5) * 0.5
  ValoresReais = np.arange(1,6)/6 #[ 0.1667, 0.3333, 0.5 , 0.6667, 0.8333]
  colors = ['Crimson','DarkGreen','SkyBlue', 'DodgerBlue', 'RoyalBlue', 'MidnightBlue']
  plt.plot(x,ValoresIniciais, '--o', color = colors[0], linewidth = 0.5, label= 'Estimativa inicial')
  plt.plot(x,ValoresReais, '--o', color = colors[1], linewidth = 0.5, label= 'Valor Real')
  for idx, epoca in enumerate(epocasPlot):
    plt.plot(x,ValoresEstadosTabela[idx], '-o', color = colors[idx+2], linewidth = 1.0, label= 'Estimativa '+str(epoca))
  plt.legend()
  plt.show()

plot_Valores()


In [ ]:
def demonstra_RandomWalk(sequencia, jogada):
  cols, rows = 7, 1
  buttons = [widgets.Button(description=' ',disabled=False,buttom_style='',
                            layout={'width': '65px', 'height': '65px'}) for i in range(cols*rows)]
  mensagem = widgets.Label(value='Jogada {:d}'.format(jogada))
  rotulo_codigo = widgets.Label()
  rotulo_jogadas = widgets.Label()
  estados = ["Festa", "A", "B", "C", "D", "E", "Aula"]
  display(widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat("+str(cols)+", 70px)")))
  display(mensagem)
  for i,e in enumerate(sequencia):
    posicao = estados.index(e)
    buttons[posicao].description = e
    time.sleep(1)
    if i != len(sequencia)-1:
      buttons[posicao].description = ' '


In [ ]:
i = 2
sequencia = EpisodioEstadosTabela[i]
jogada = i
demonstra_RandomWalk(sequencia=sequencia, jogada = jogada)


# TD-Learning

O aprendizado por diferença temporal (DT) é um método baseado em valores. Ou seja, visa aprender uma função valor que represente o retorno esperado (sequência de recompensa descontada cumulativa) dado que as ações são selecionadas de acordo com alguma política $\pi$:

$$
G_t = R_{t+1}+\gamma R_{t+2} + \gamma^2 R_{t+3} +\cdots
$$

$$
v_\pi=\mathbb{E}_\pi\left[G_t|s=S_t\right]
$$

O aprendizado por TD estima o retorno executando uma ação, amostrando uma recompensa e, em seguida, inicializando sua estimativa atual de qual será o retorno do próximo estado em diante. Isso se chama, em RL, *bootstrapping*, isto é, atualização de um valor com base em algumas estimativas e não em valores exatos. Portanto, o método permite que passo seja dado na direção dessa estimativa, sem ter que esperar por toda a sequência de recompensas:
$$
\hat G_t = R_{t+1}+\gamma R_{t+2} + \gamma^2 R_{t+3} +\cdots
$$
$$
v(S_t) = V(S_t) + \alpha\left[G_t - v(S_t)\right]
$$

Por essa razão, o aprendizado por TD é visto como um algoritmo intermediário que unifica o aprendizado Monte Carlo, onde as recompensas são amostradas, e a programação dinâmica, onde você inicializa as estimativas atuais.

__Atualizações incrementais da avaliação da política por Monte Carlo:__
$$
v(S_t) = V(S_t) + \alpha\left[\color{red}{G_t - v(S_t)}\right]
$$
__Atualizações de avaliação de política por TD(0):__
$$
v(S_t) = v(S_t) + \alpha\left[ \color{blue}{R_{t+1} + \gamma v(S_{t+1}) - V(S_t)}\right]
$$



## O que é bootstrapping?
Bootstrapping é uma técnica de reamostragem que ajuda a estimar a incerteza de um modelo estatístico.

Inclui amostragem do conjunto de dados original com reposição e geração de vários novos conjuntos de dados do mesmo tamanho que o original.

Cada um desses novos conjuntos de dados é usado para calcular a estatística desejada, como a média ou o desvio padrão.

Esse processo é repetido várias vezes e os valores resultantes são usados para construir uma distribuição de probabilidade para a estatística desejada.

Essa técnica é frequentemente usada em aprendizado de máquina para estimar a precisão de um modelo, validar seu desempenho e identificar áreas que precisam ser melhoradas.

#### Bootstrap: como estimar a média com poucas amostras
Exemplo extraído do [link](https://www.kdnuggets.com/2023/03/bootstrapping.html).

Vamos calcular a média de altura de nossos monstros estudantes. Como eles são em 10mil, amostraremos 200 alunos e usaremos bootstrapping.

<center><img src='https://drive.google.com/uc?export=view&id=1PKK6VqBBW8jKXW92VGcz4ZNAJlGJ6yOV' width="600"></center>

In [ ]:
x = np.random.normal(loc= 325.0, scale=1.0, size=10000)
np.mean(x)


Calculamos a média da altura de uma amostra grande, com 10.000 alunos.

Agora, imagine que só temos acesso a uma amostra bem menor, com apenas 200 alunos. Como podemos usar esses dados para estimar a média da altura da população?

Uma técnica muito útil para isso é chamada de bootstrap. A ideia é simples: a partir da nossa amostra, criamos várias novas amostras, sorteando os alunos com reposição.

O que significa com reposição?
Cada vez que escolhemos um aluno, ele continua disponível para ser escolhido novamente. Então, é possível (embora improvável) que um mesmo aluno apareça mais de uma vez na mesma amostra. Isso ajuda a simular diferentes "cenários" da população.

No nosso exemplo, vamos gerar 40 novas amostras, cada uma com 5 alunos, a partir da amostra original.

Para cada uma dessas 40 amostras:
* sorteamos 5 alunos com reposição,
* calculamos a média da altura desses 5 alunos,
* guardamos essa média em uma lista chamada sample_mean.

Por fim, calculamos a média das 40 médias que obtivemos. Esse valor final será a nossa nova estimativa da média da população.



In [ ]:
sample_mean = []

# Bootstrap Sampling
for i in range(40):
    y = random.sample(x.tolist(), 5)
    avg = np.mean(y)

    sample_mean.append(avg)
np.mean(sample_mean)


## Voltemos ao nosso exemplo: Random Walk com 5 estados

Vamos implementar TD(0) para estimar $v_\pi$ do tradicional problema *Random Walk*.

<center><img src='https://drive.google.com/uc?export=view&id=1I5jIH-k-6Pc7dFDA3LF9ZZXiLu1Uvtvk' width="700"></center>



* Todos os episódios começam no estado central 𝐶;
* O agente vai para a esquerda ou para a direita, com igual probabilidade;
* Os episódios terminam na extrema esquerda (festa) ou na extrema direita (aula);
* Quando um episódio termina à direita, na aula, ocorre uma recompensa de 1; todos outras recompensas são zero;
* Como esta tarefa não é descontada e é episódica, o verdadeiro valor de cada estado é a probabilidade de terminar à direita se começar daquele estado: os valores verdadeiros de todos os estados, de 𝐴 a 𝐸, são $v_\pi = \begin{bmatrix}1/6 & 2/6 & 3/6 & 4/6 & 5/6\end{bmatrix}$.



Ou seja, a resposta final do exercício deve ser:
`ValoresEstadosdos[1:-1] = [ 0.1667,  0.3333,  0.5   ,  0.6667,  0.8333]`


In [ ]:
class RandomWalk_TD():
    gamma = 1.0      # Fator de desconto para recompensas futuras (=1 porque o jogo é finito e determinístico)

    def __init__(self, alpha = 0.10, valores = np.ones(7)*0.5):
        self.estados = ["Festa", "A", "B", "C", "D", "E", "Aula"]      # Todos os estados
        self.posicao = 3                                              # Posição inicial é "C"
        self.estado = self.estados[self.posicao]
        self.valores = valores
        self.end = False
        self.alpha = alpha

    def escolhe_acao(self):
        if not self.end:
            acao = np.random.choice(["esquerda", "direita"])

            if acao == "esquerda":
                #Movimento para Esquerda
                nova_posicao = self.posicao - 1
            elif acao == "direita":
                #Movimento para Direita
                nova_posicao = self.posicao + 1

            recompensa = self.get_recompensa(nova_posicao)

            self.TD0(self.posicao, nova_posicao)

            self.posicao = nova_posicao
            self.estado = self.estados[self.posicao]

            if (self.estado == "Festa") or (self.estado == "Aula"):
                #Random Walk acabou!
                self.end = True
        else:
            print("A movimentação não é possível porque o Random Walk já terminou.")

    def get_recompensa(self, nova_posicao):
        return 1.0 if self.estados[nova_posicao] == "Aula" else 0.0

    def TD0(self, antiga_posicao, nova_posicao):
        target = self.get_recompensa(nova_posicao) + self.gamma * self.valores[nova_posicao]
        delta = target - self.valores[antiga_posicao]
        self.valores[antiga_posicao] += self.alpha * delta


In [ ]:
ValoresEstados = np.ones(7) * 0.5              # estimativas inicializadas em 0.5
ValoresEstados[0] = 0
ValoresEstados[6] = 0
max_episodios  = 101
epocasPlot = [0,1, 10,  100]
ValoresEstadosTabela = []

for epoca in range(max_episodios):
    Mike = RandomWalk_TD(valores = ValoresEstados)    # Usa estimativas de valor da iteração anterior
    while not Mike.end:
        Mike.escolhe_acao()
    ValoresEstados = copy.deepcopy(Mike.valores)                # Atualiza as estimativas de valor
    if epoca in epocasPlot:
      ValoresEstadosTabela.append(ValoresEstados[1:-1])

print(ValoresEstados[1:-1])


In [ ]:
def plot_graphValores():
  x = ['A','B', 'C','D', 'E']
  ValoresIniciais = np.ones(5) * 0.5
  ValoresReais = np.arange(1,6)/6 #[ 0.1667, 0.3333, 0.5 , 0.6667, 0.8333]
  colors = ['Crimson','DarkGreen','SkyBlue', 'DodgerBlue', 'RoyalBlue', 'MidnightBlue']
  plt.plot(x,ValoresIniciais, '--o', color = colors[0], linewidth = 0.5, label= 'Estimativa inicial')
  plt.plot(x,ValoresReais, '--o', color = colors[1], linewidth = 0.5, label= 'Valor Real')
  for idx, epoca in enumerate(epocasPlot):
    plt.plot(x,ValoresEstadosTabela[idx], '-o', color = colors[idx+2], linewidth = 1.0, label= 'Estimativa '+str(epoca))
  plt.legend()
  plt.show()

plot_graphValores()


In [ ]:
alpha_list = [0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2]
max_episodios  = 100
ValoresReais = [ 0.1667, 0.3333, 0.5 , 0.6667, 0.8333]
n_experiments = 100
mse = np.zeros((n_experiments, len(alpha_list), max_episodios+1))


In [ ]:
for exp in range(n_experiments):
  if (exp + 1) % 20 == 0:
    print('Experimento {0}'.format(exp + 1))
  for i, alpha in enumerate(alpha_list):
    ValoresEstados = np.ones(7) * 0.5              # estimativas inicializadas em 0.5
    ValoresEstados[0] = 0
    ValoresEstados[6] = 0
    max_episodios  = 101
    epocasPlot = [0, 1, 10,  100]
    ValoresEstadosTable = []

    for epoch in range(max_episodios):
      Mike = RandomWalk_TD(alpha = alpha, valores = ValoresEstados)    # Usa estimativas de valor da iteração anterior
      while not Mike.end:
          Mike.escolhe_acao()
      ValoresEstados = copy.deepcopy(Mike.valores)                # Atualiza as estimativas de valor
      if epoch in epocasPlot:
        ValoresEstadosTable.append(ValoresEstados[1:-1])
      mse[exp, i,epoch] = mean_squared_error(ValoresReais, ValoresEstados[1:-1])


In [ ]:
rms = np.sqrt(mse)
rms.shape


In [ ]:
rms_ave = rms.mean(axis=0)
rms_std = rms.std(axis=0)
rms_ave.shape


In [ ]:
for i, alpha in enumerate(alpha_list):
#    vals = rms_ave[:,i]
    plt.plot(range(101),rms_ave[i,:], '-', label=r'$\alpha$= {0}'.format(alpha))

plt.legend(loc='best');
plt.xlabel('Episódios');
plt.ylabel('RMS');
plt.ylim(0.05, 0.26);


O método TD (Temporal Difference) nos permite aprender *online* – ao mesmo tempo que interagimos com um ambiente – e baseia-se na noção de bootstrapping. Isso significa que usamos nossa aproximação atual para o valor de um estado (que pode estar errado) para atualizar nosso valor estimado para outro estado.

Tudo vai bem desde que todas as aproximações melhorem com o tempo. Esse método é chamado de TD(0) e é viesado, embora tenha variância reduzida. Um método de estimativa de Monte-Carlo não é viesado, mas tem muita variação, pois usa o resultado de um episódio completo para realizar uma atualização. A variação vem do fato de que, a cada interação, há aleatoriedade envolvida na escolha de uma ação – no caso de uma política estocástica – e do fato de que a dinâmica do ambiente também é aleatória (lembre-se, temos uma distribuição sobre os possíveis próximos estados, que depende do estado atual e da ação tomada).

Um problema com TD(0) é que ele usa informações de apenas uma etapa para realizar uma atualização. Imagine que você está jogando um jogo e toma uma decisão ruim no começo (ex.: virou à esquerda em vez da direita). Essa decisão só vai te dar uma recompensa ruim várias jogadas depois. O TD(0) só olha para a última recompensa imediata para aprender. Ou seja, ele não consegue "ligar" a recompensa ruim às ações passadas que a causaram. Ele só atualiza o valor do estado anterior imediato, ignorando que o erro veio de muito antes.

Isso faz com que o aprendizado seja lento, porque, se a recompensa ruim veio de uma ação 10 passos atrás, o TD(0) só vai ajustar aos poucos, passo a passo. Ele não "propaga" o erro rapidamente para trás no tempo.

Para resolver esse problema, podemos pensar que basta usar mais de uma etapa para realizar uma atualização.

<center><img src='https://drive.google.com/uc?export=view&id=1MO_G8b-3vPS6rnSaf7EhFVE5UyjMznH4' width="400"></center>


## N-steps

$$
n = 1 \rightarrow G_t^{(1)} = r_{t+1} + \gamma {V_\pi}(s_{t+1}) \\
n = 2 \rightarrow G_t^{(2)} = r_{t+1} + \gamma r_{t+2} + \gamma^2 {V_\pi}(s_{t+2}) \\
\vdots \\
\forall n \rightarrow G_t^{(n)} = r_{t+1} + \gamma r_{t+2} + \dots + \gamma^{n-1} r_{t+n} + \gamma^n {V_\pi}(s_{t+n})
$$
onde a última parcela é sempre estimada, enquanto as demais são observadas. A função valor é então atualizada da seguinte forma:
$$
V(S_t) \leftarrow V(S_t) + \alpha \left[G_t^{(n)}-V(S_t)  \right]
$$


## TD ($\lambda$)

Mas naturalmente surge a seguinte pergunta: como escolhemos o número $n$ de etapas (steps) a serem usadas? Escolher $n$ pode ser difícil e certamente não generalizará para diferentes ambientes, então vamos encontrar uma maneira de evitar escolher $n$ completamente! __Faremos isso escolhendo todos os valores diferentes de n de uma só vez.__ Como?

Vamos ponderar o retorno de $n$ etapas $G^{(n)}_t$ usando um peso que decai exponencialmente com o tempo. Isso é feito introduzindo um fator $\lambda \in [0,1]$ e ponderando o n-ésimo retorno com $\lambda^{n-1}$. Como queremos que todos esses pesos somem um (para ter uma média ponderada), precisamos normalizá-los com,
$$
\sum_{n=1}^\infty \lambda^{n-1} = \sum_{n=0}^{\infty} \lambda^n = \frac{1}{1 - \lambda}
$$

Portanto, a constante de normalização que procuramos é $(1−\lambda)$. Isso dá origem à definição do $\lambda$-retorno:
$$
G_t^\lambda = (1 - \lambda) \sum_{n=1}^\infty \lambda^{n-1} G_t^{(n)}
$$

<center><img src='https://drive.google.com/uc?export=view&id=1CV3wDisMMnDJNW2tmlOOO5ab1x-ffdiK' width="400"></center>


A fórmula do TD($\lambda$) para o retorno $G_t^\lambda$ é dada por:

$$
G_t^\lambda = (1-\lambda) \sum_{n=1}^{T-t-1} \lambda^{n-1} G_t^{(n)} + \lambda^{T-t-1} G_t^{(T-t)}
$$

onde:
*  $T$ é o tempo final do episódio,
*  $t$ é o tempo atual,
*  $G_t^{(n)}$ é o retorno $n$-passos a partir do tempo $t$,
*  $\lambda \in [0,1]$ é o parâmetro que controla a média ponderada dos retornos.

Cada retorno $n$-passos tem peso

$$
w_n = (1-\lambda) \lambda^{n-1}, \quad n=1,2,\ldots, T - t - 1
$$

Assim, os retornos até $T-t-1$ são ponderados por uma série geométrica que decresce conforme $n$ aumenta.

Somando os pesos parciais temos

$$
S = \sum_{n=1}^{T-t-1} w_n = (1-\lambda) \sum_{n=1}^{T-t-1} \lambda^{n-1} = (1-\lambda) \frac{1-\lambda^{T-t-1}}{1-\lambda} = 1 - \lambda^{T-t-1}
$$

Para garantir que a soma dos pesos seja 1, o peso do último retorno $G_t^{(T-t)}$ deve ser

$$
w_{\text{último}} = 1 - S = \lambda^{T-t-1}
$$

O retorno $G_t^{(T-t)}$ é o retorno total até o final do episódio (ou retorno Monte Carlo). Isso porque, a partir de $n = T-t$, os retornos $n$-passos coincidem com o retorno completo $G_t^{(T-t)}$.

Portanto, o peso da última parcela representa a soma dos pesos para todos os retornos $n$-passos maiores ou iguais a $T-t$, que é uma série geométrica infinita com razão $\lambda$.


Veja que,

Para $\lambda = 0$:
$$
G_t^\lambda = G_t^{(1)},
$$
ou seja, considera-se apenas o retorno de um passo à frente, o que corresponde ao método TD(0).

Para $\lambda = 1$:
$$
G_t^\lambda = G_t^{(T-t)},
$$
ou seja, considera-se apenas o retorno total do episódio, equivalente ao método Monte Carlo.


Assim, a fórmula do TD($\lambda$) combina todos os retornos $n$-passos em uma média ponderada, onde o parâmetro $\lambda$ regula a importância relativa entre retornos mais curtos (próximos a TD(0)) e o retorno completo (equivalente a Monte Carlo).


In [ ]:
lambdas_lista = [0.,0.01,0.5,0.75,0.9,0.95,0.99,1.0]
n_lista = np.arange(0,10,0.001)
for lamb in lambdas_lista:
  y=[]
  for n in n_lista:
    y.append((1.-lamb)*lamb**n)
  plt.plot(n_lista,y, '-', label=r'$\lambda$= {0}'.format(lamb))
plt.legend(loc='best');
plt.xlabel(r'$n$');
plt.ylabel(r'$\lambda^n(1-\lambda)$');
plt.ylim(0, 1.1);


In [ ]:
lambdas_list = [0.,1.0]
n_list = np.arange(0,70,0.1)
for lamb in lambdas_list:
  y=[]
  for n in n_list:
    y.append((1.-lamb)*lamb**n)
  plt.plot(n_list,y, '-', label=r'$\lambda$= {0}'.format(lamb))
plt.legend(loc='best');
plt.xlabel(r'$n$');
plt.ylabel(r'$\lambda^n(1-\lambda)$');
plt.ylim(0, 1.1);


Podemos ver como diferentes valores de $\lambda$ afetam o valor inicial de um retorno e como esse valor decai com o tempo. Valores maiores de lambda levam a um decaimento mais lento (a informação distante do instante atual $t$ recebe uma importância não desprezível).




### $\lambda$-return: Forward view

Com  $G_t^\lambda$ definido anteriormente, pode-se atualizar a função valor como,
$$
V(S_t) \leftarrow V(S_t) + \alpha \left[G_t^{\lambda}-V(S_t)  \right]
$$

Veja que para $\lambda=1$ tem-se o método de Monte Carlo, também conhecido como TD(1), e para $\lambda = 0$ recupera-se o método TD(0).



#### Qual o problema do método?

O problema é que o $\lambda$-retorno envolve __todos os possíveis retornos__ de $n$ passos e, como consequência, envolve informações de cada passo de tempo de nossa trajetória. Voltamos às atualizações episódicas: precisamos esperar que uma trajetória seja concluída e, então, poderemos calcular todos os possíveis retornos de $n$ etapas e combiná-los usando a média ponderada por $\lambda$.

Esse tipo de TD($\lambda$) que vimos acima é chamado de *forward view* ou visão para frente. Mas esse esquema de ponderação por $\lambda$ nos levará a um *truque* que permite atualizações online enquanto é virtualmente equivalente à intuição que derivamos acima. Para fazer isso, usaremos algo chamado *rastreamento de elegibilidade* ( em inglês, *eligibility trace*). __ Mas isso é assunto para próxima aula__.

# Controle

Como aprendemos, no domínio do aprendizado por reforço, os métodos de controle podem ser divididos em duas abordagens principais: model-based e model-free. Enquanto a primeira requer um modelo completo do ambiente, a segunda, mais flexível e amplamente aplicada, aprende diretamente da interação com o ambiente sem tal modelo.

Nos métodos de controle em aprendizado por reforço do tipo model-free, o agente busca aprender como agir no ambiente sem conhecer previamente como o ambiente funciona. Ou seja, ele não tem um modelo das transições entre estados ou da dinâmica que gera as recompensas. O que ele possui é a capacidade de experimentar: ele executa ações, observa os resultados e, com base nisso, ajusta sua política ou sua função de valor. Assim, o controle acontece de forma reativa — o agente vai aprimorando seu comportamento diretamente a partir da experiência acumulada, sem precisar simular possíveis futuros.

Dentro do paradigma model-free, as estratégias se diferenciam principalmente pela forma como lidam com a política de exploração, dando origem às abordagens *on-policy* e *off-policy*. Nos métodos *on-policy*, o agente avalia e aprimora exatamente a mesma política que está usando para tomar decisões. É um aprendizado "on the job" (aprender fazendo). Isso cria um ciclo natural de exploração e adaptação, mas também exige que o agente explore constantemente para não ficar preso em comportamentos subótimos. Já nos métodos *off-policy*, a situação é diferente: o aprendizado acontece sobre uma política-alvo, mas as experiências são coletadas a partir de outra política, que pode ser mais exploratória ou mais segura. Assim, é possível aprender sobre a política ideal observando comportamentos alternativos, o que permite separar o processo de coleta de dados do processo de otimização da política.

No cerne dessas estratégias está o delicado equilíbrio entre exploração e experimentação: como garantir que o agente experimente suficientemente o ambiente para não se limitar a um comportamento inicial pobre? $\varepsilon$-soft é uma estratégia de exploração que ajuda o agente a não ficar preso em ações que parecem boas só no começo, dando uma chance de explorar outras ações.

Uma política ε-soft garante que, para todo estado $s \in \mathcal{S} $ e toda ação  $ a \in \mathcal{A} $:

$$
\pi(a \mid s) \geq \frac{\varepsilon}{|A(s)|} > 0,
$$

onde:

- $ \varepsilon > 0 $ (ex.: $ \varepsilon = 0.1 $) controla o mínimo de exploração.
- $ |A(s)| $ é o número de ações possíveis no estado $ s $.

Todas as ações têm probabilidade não nula de serem escolhidas, evitando que o agente pare de experimentar completamente (útil em ambientes estocásticos ou com recompensas incertas). Apesar da exploração forçada, a política pode se tornar arbitrariamente próxima de uma política determinística ótima, por exemplo, atribuindo  
$$
\pi(a^* \mid s) \approx 1 - \varepsilon
$$
para a ação ótima $ a^* $.

Uma versão particular muito usada de $\varepsilon$-soft, é a estratégia $\varepsilon$-greedy. Nessa abordagem, o agente segue a melhor ação conhecida com alta probabilidade (por exemplo, $1-\varepsilon$), mas de tempos em tempos (com probabilidade $\varepsilon$) escolhe uma ação aleatória. Isso garante um equilíbrio entre experimentação (descobrir novas possibilidades) e exploração (aproveitar o que já se sabe que funciona bem). Tanto em métodos on-policy quanto off-policy, o uso de $\varepsilon$-greedy é uma ferramenta prática para garantir diversidade na experiência coletada, permitindo que o aprendizado model-free seja mais robusto e eficaz.


| Política   | Definição                                                                 | Como Escolhe Ações?                                                                 | Exemplo (ε = 0.1, m=4 ações)                        |
|------------|---------------------------------------------------------------------------|------------------------------------------------------------------------------------|-----------------------------------------------------|
| ε-gulosa   | - Ação gulosa (melhor valor) tem 1 − ε + ε/m de chance.<br>- Outras ações têm ε/m cada. | - Exploração uniforme: ações não-gulosas têm a mesma chance pequena (ε/m).<br>- Exploração concentrada: ação gulosa domina. | Melhor ação: 92.5% (1 − 0.1 + 0.025).<br>Outras: 2.5% cada. |
| ε-soft     | - Todas as ações têm probabilidade ≥ ε/m.<br>- Pode distribuir probabilidade desigualmente (não precisa ser gulosa). | - Mais flexível: pode favorecer ações quase-gulosas mais que as piores.<br>- Ainda garante mínimo ε/m para todas. | Ações podem ter: 70%, 20%, 5%, 5% (desde que cada uma ≥ 2.5%). |

# Melhoria de Política em model-free

Com um modelo, apenas os valores de estado são suficientes para determinar uma política; basta olhar um passo à frente e escolher qualquer ação que leve à melhor combinação de recompensa e próximo estado, como fizemos no capítulo sobre DP. Sem um modelo, no entanto, os valores de estado por si só não são suficientes.

Deve-se estimar explicitamente o valor de cada ação para que os valores sejam úteis para sugerir uma política.


<center><img src='https://drive.google.com/uc?export=view&id=1Ww3dYWpmRGYvG1RRDroW3HUg-NGWhtoE' width="600"></center>



## Teorema

Para qualquer política $\pi$, a política $\varepsilon$-gulosa $\pi'$ em relação à função de valor de ação $Q_\pi$ é melhor ou igual à política original $\pi$ em termos de valor esperado do estado, ou seja, $V_{\pi'}(s) \geq V_\pi(s)$ para todos os estados $s \in \mathcal{S}$.

### Prova: A política $\varepsilon$-gulosa melhora ou mantém o valor esperado

Queremos provar que, para qualquer política $\pi$, a política $\varepsilon$-gulosa $\pi'$ em relação à função de valor de ação $Q_\pi$ é pelo menos tão boa quanto $\pi$, ou seja:

$$
V_{\pi'}(s) \geq V_{\pi}(s), \quad \forall s \in \mathcal{S}.
$$

onde $Q_{\pi}(s,a)$ é valor da ação $a$ no estado $s$ seguindo a política $\pi$ e $\pi'$ é política $\varepsilon$-gulosa em relação a $Q_{\pi}$.

A política $\varepsilon$-gulosa é definida como:

$$
\pi'(a|s) =
\begin{cases}
1 - \varepsilon + \frac{\varepsilon}{m(s)} & \text{se } a = a^* = \arg\max_{a} Q_{\pi}(s,a), \\
\frac{\varepsilon}{m(s)} & \text{caso contrário}.
\end{cases}
$$
onde $m(s)$ é o número de ações possíveis no estado $s$.

O valor esperado da política $\pi'$ para o estado $s$, usando a função $Q_{\pi}$, é:
$$
V_{\pi'}(s) = \sum_a \pi'(a|s) Q_{\pi}(s,a) = \left(1-\varepsilon + \frac{\varepsilon}{m(s)}\right) Q_{\pi}(s,a^*) + \sum_{a \neq a^*} \frac{\varepsilon}{m(s)} Q_{\pi}(s,a)
$$

Sabemos que:

$$
V_{\pi}(s) = \sum_a \pi(a|s) Q_{\pi}(s,a) = \left(1-\varepsilon + \frac{\varepsilon}{m(s)}\right) Q_{\pi}(s,a) + \sum_{a \neq a^*} \frac{\varepsilon}{m(s)} Q_{\pi}(s,a)
$$

Como $a^*$ é a ação ótima para $Q_{\pi}(s,a)$, temos:

$$
Q_{\pi}(s,a^*) \geq Q_{\pi}(s,a), \quad \forall a.
$$
portanto,

$$
\left(1-\varepsilon + \frac{\varepsilon}{m(s)}\right) Q_{\pi}(s,a^*) \geq \left(1-\varepsilon + \frac{\varepsilon}{m(s)}\right) Q_{\pi}(s,a)
$$
de modo que
$$
V_{\pi'}(s) \geq V_{\pi}(s), \quad \forall s \in \mathcal{S}.
$$

Assim, a política $\varepsilon$-gulosa $\pi'$:
- favorece a ação ótima $a^*$ com alta probabilidade;
- assegura que o valor esperado do estado não diminui em relação à política $\pi$;
- portanto, garante que $V_{\pi'}(s) \geq V_{\pi}(s), \forall s$.


# SARSA

Os métodos de Monte Carlo para estimativa da política ótima usam uma política $\pi$ "soft" (Sejam "$\varepsilon$-gulosas" como no exercício do jogo-da-velha ou políticas que exploram diferentes ações iniciais, como no caso do Monte Carlo Exploring Starts) para explorar o espaço de ações e estimar os valores de $Q_\pi(s,a)$.

Em seguida eles criam uma nova política, baseada nas novas estimativas de $Q_\pi(s,a)$ e repetem o processo.

Nos métodos de Monte Carlo, a atualização das estimativas dos valores de $Q$ ocorre *somente após o término de cada episódio*.

Em contrapartida, os métodos de diferenças temporais atualizam as estimativas de $q$ *a cada passo*.

Vamos recapitular as equações de recorrência para valores de estado:

\begin{equation}
v_{\pi}(s) = \sum_A \pi(A) \sum_{r,s'} p(r,s' | s,a) \left(r + \gamma v_\pi(s')\right)
\end{equation}

Esta é uma equação que define $v_\pi(s)$ em função de todos os valores dos possíveis estados sucessores $s'$.

Considerando-se a definição de valor do par ação-estado:

\begin{align}
q_{\pi}(s,a) &= \sum_{r\in \mathcal{R},s'\in \mathcal{S}} p(r,s'| s,a) \left(r + \gamma v_\pi(s')\right)\\
v_{\pi}(S) &= \sum_{a \in \mathcal{A}} \pi(a|s) q(s,a)
\end{align}

Por esta definição, é possível definir uma equação recorrente o valor de um par estado-ação $q(S, A)$ em função dos valores de todos os possíveis pares seguintes de estados-ações:

\begin{equation}
q_{\pi}(s,a) = \sum_{r,s'} p(r,s' | s,a) \left(r + \gamma \sum_{a'} \pi(a'|s') q_\pi(s',a')\right)
\end{equation}

Esta equação motiva o algoritmo SARSA.

A ideia é aprimorar as estimativas para os valores de $q_\pi(s,a)$ usando uma política "$\varepsilon$-gulosa".

Considere que em uma etapa da exploração o agente estava no estado $s$, tomou a ação $a$, recebeu a recompensa $r$ e acabou no estado $s'$.
Na etapa seguinte, o agente tomou a ação $a'$.

Então, se $Q_\pi(s,a)$ é uma estimativa para $q_\pi(s,a)$, ela pode ser atualizada com:

\begin{equation}
Q_\pi(s,a) \leftarrow Q_\pi(s,a) + \alpha \left(r + \gamma Q_\pi(s',a') - Q_\pi(s,a) \right)
\end{equation}

onde $\alpha \leq 1$ é uma *taxa de aprendizado*.

Portanto, SARSA é um algoritmo on-policy onde, no estado atual, s, uma ação, a, é realizada e o agente recebe uma recompensa, r, e acaba no próximo estado, s', e realiza a ação, a', em s'.

Note que o SARSA usa *dois* pares de estado-ação para fazer esta atualização.
Ele atualiza a estimativa do par $(s,a)$ usando o valor da recompensa na transição de $S$ para $S'$ e a estimativa do par $(s', a')$.
Daí o seu nome, "SARSA": da tupla (s, a, r, s', a').
<center><img src='https://drive.google.com/uc?export=view&id=1FQ2hW6VnRUsbIeqmTUUjHT72Z7fuxT7E' width="100"></center>

É utilizada uma tabela, definida como Tabela Q, inicializada em 0. As colunas referem-se às ações possíveis e as linhas aos estados. Durante o treinamento, a tabela 𝑄 é otimizada. Então nosso agente pode usar as informações armazenadas nesta tabela para escolher a melhor ação em cada estado.


Q-table | Ação (0) | Ação (1)  |   ...    | Ação (m-1) | Ação (m)
--------|----------|-----------|----------|------------|----------
0       | 0        | 0         |         | 0          | 0        
1       | 0        | 0         |         | 0          | 0
2       | 0        | 0         |         | 0          | 0
...     | 0        | 0         |         | 0          | 0
n-1     | 0        | 0         |         | 0          | 0
n       | 0        | 0         |         | 0          | 0

SARSA usa a abordagem de Diferença Temporal (TD), então o algoritmo continuará atualizando a tabela Q após cada passo até atingirmos o número máximo de iterações ou a solução convergir para uma ótima. O algoritmo abaixo será implementado a seguir, para o problema do *grid com vento*.

<center><img src='https://drive.google.com/uc?export=view&id=1pPts7h-5Z03hj_thnYjfqRfimkzCJ51y' width="600"></center>Fonte: Richard S. Sutton and Andrew G. Barto, Reinforcement Learning: An Introduction, 2nd ed, The MIT Press.
Pág. 130


Para o SARSA efetivamente estimar a política ótima, é necessário que a política "soft" gradativamente torne-se determinística.
Isso é feito rodando-se várias vezes o SARSA, a cada uma baixando-se o valor do parâmetro $\varepsilon$.

## Windy Gridworld

Windy Gridworld mostra um gridworld padrão, com estados iniciais (S) e objetivos (G), mas com uma diferença: há um vento cruzado ascendente no meio da grade. As ações são as quatro padrão - para cima, para baixo, para a direita e para a esquerda - mas na região intermediária os próximos estados resultantes são deslocados para cima por um "vento", cuja força varia de coluna para coluna. A força do vento é dada abaixo de cada coluna, em número de células deslocadas para cima. Por exemplo, se você estiver uma célula à direita da meta (G), a ação à esquerda o levará à célula logo acima da meta. Tratemos isso como uma tarefa episódica não descontada, com recompensas constantes até que o estado objetivo seja alcançado.

<center><img src='https://drive.google.com/uc?export=view&id=1f4C0tSZXU18FIVe034isCc9pfqTl7vb1' width="400"></center>





O código do ambiente, classe `gridWorld` definida abaixo foi adaptado do [link](https://www.kaggle.com/code/kamal007/windy-grid-world-solution-using-sarsa-q-learning/notebook).

Os atributos da classe são:
* `start`: estado inicial
* `goal`: estado final (terminal)
* `row`: número de linhas do grid
* `col`: número de colunas do grid
* `x_max = col-1`: número máximo, com referência à coluna que uma ação pode levar
* `y_max`: número máximo, com referência à linha, que uma ação pode levar
* `wind_1`: estados em que o vento leva 1 linha para cima
* `wind_2`: estados em que o vento leva e linhas para cima
* `actions_list = ['N', 'L', 'S', 'O']`: lista de ações possíveis: Norte,Leste,Sul,Oeste referentes, respectivamente, a Cima,Direita,Baixo,Esquerda.

Os métodos `coord_state` e `state_coord` transformam, respectivamente, coordenada em estado e estado em coordenada. Por exemplo, o estado inicial da Figura (S) tem coordenadas (3,0) e refere-se ao estado 30. Na função `coord_state` tem-se que:
```
state = 0 + 10 * 3 = 30
```
e na função `state_coord` tem-se que:
```
x = 30 % 10 = 0
y = (30 - 0) / 10 = 3
```

O método `setTerminal` é usado para definir os estados iniciais e finais em coordenadas, que serão transformadas em estados.

O método `nextState` define o estado seguinte a partir da ação e do estado atual, o método `checkterminal` verifica se o episódio chegou ao fim, e o método `rewardFunction` atribui uma recompensa imediata à ação tomada.


In [ ]:
class gridWorldWindy:

    def __init__(self):
        self.start = 0
        self.goal = 0
        # configuração das linhas e colunas do gridWorld
        self.row = 7
        self.col = 10
        self.x_max = self.col - 1
        self.y_max = self.row - 1
        # colunas com vento e seus efeitos
        self.wind_1 = [3, 4, 5, 8]
        self.wind_2 = [6, 7]
        # Fornecer lista de ações:
        self.actions_list = ['N', 'L', 'S', 'O']

    def coord_state(self,pos):
        return pos[1] + self.col * pos[0]

    def state_coord(self,state):
        # O parâmetro 'state' é um número inteiro que representa a posição na grade
        x = state % self.col
        y = (state - x) / self.col
        return x,y

    def setTerminal(self, startState, goalState):
        # startState e goalState são tuplas
        self.start = self.coord_state(startState)
        self.goal = self.coord_state(goalState)

    def nextState(self, state, action):
        x,y = self.state_coord(state)
        # Ações
        del_x = 0
        del_y = 0
        if action == 'L':
            del_x = 1
        elif action == 'O':
            del_x = -1
        elif action == 'N':
            del_y = -1
        elif action == 'S':
            del_y = 1
        else:
            raise('Ação inválida! As ações devem estar em: ',self.actions_list)
        # Movimento para a nova posição
        new_x = max(0, min(x + del_x, self.x_max))
        new_y = max(0, min(y + del_y, self.y_max))
        # Quando o vento sopra para um novo estado...
        if new_x in self.wind_1:
            new_y = max(0, new_y - 1)
        if new_x in self.wind_2:
            new_y = max(0, new_y - 2)

        return self.coord_state((new_y,new_x))

    def checkTerminal(self, state):
        return state == self.goal

    def rewardFunction(self, state_prime):
        # Quando atinge o estado Termonal (G), a recompensa é 0,
        # caso contrário, todas as ações retornam uma recompensa de -1
        if state_prime == self.goal:
            return 0
        else:
            return -1


A função `trajectoryPath` desenha o trajeto do agente durante o episódio.  No exemplo abaixo, veja que o agente foi  *empurrado*  pelo vento nos passos 4-10 e depois, novamente, nos passos 18-19.

```
[[ 0.  0.  0.  0.  6.  7.  8.  9. 10. 11.]
 [ 0.  0.  0.  0.  5.  0.  0.  0.  0. 12.]
 [ 0.  0.  0.  4.  0.  0.  0.  0.  0. 13.]
 [ 1.  2.  3.  0.  0.  0.  0. 19.  0. 14.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0. 15.]
 [ 0.  0.  0.  0.  0.  0.  0.  0. 18. 16.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0. 17.]]
```

Se o agente passar duas vezes pelo mesmo estado, apenas a última visita é computada na respectiva coordenada. Verifique que os estados $6,10,11$ não aparecem na trajetória abaixo. O estado $6$ foi sobrescrito pelo $7$ e os estados $10,11$ foram sobrescritos pelo estado $12$.


```
[[ 0.  0.  0.  0.  0. 12. 13. 14. 15. 16.]
 [ 0.  0.  0.  0.  9.  0.  0.  0.  0. 17.]
 [ 0.  0.  0.  8.  0.  0.  0.  0.  0. 18.]
 [ 1.  0.  0.  0.  0.  0.  0. 25.  0. 19.]
 [ 2.  3.  0.  7.  0.  0.  0.  0.  0. 20.]
 [ 0.  4.  5.  0.  0.  0.  0.  0. 24. 22.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0. 23.]]
```



In [ ]:
def trajectoryPath(world, traj):
    # Inicializa gridworld
    world_map = np.zeros((world.row, world.col))
    for i,state in enumerate(traj):
        x = int(state % world.col)
        y = int((state - x) / world.col)
        world_map[y, x] = i + 1
    wm = '\n'.join([''.join(['{:4d}'.format(item) for item in row]) for row in world_map.astype(int)])
    print(wm)
    print("\n")


A função `gridWorld_SARSA` possui o método SARSA aplicado ao problema que acabamos de definir com a classe `gridWorldWindy`. Possui `epsGreedy` e `update_Q` como funções *nested*. A primeira define a ação de acordo com a política $\varepsilon$-greedy e a segunda atualiza a tabela Q.

Na função `gridWorld_SARSA`, dentro do loop em número de episódios (`while ep < max_episodes:`) existe um loop sobre os estados (`while not world.checkTerminal(state):`) que termina quando o estado Terminal é alcançado. Dentro do loop mais interno, o método SARSA é aplicado:


```
# 𝑄(𝑆𝑡,𝐴𝑡)=𝑄(𝑆𝑡,𝐴𝑡)+𝛼[𝑅𝑡+1+𝛾𝑄(𝑆𝑡+1,𝐴𝑡+1)−𝑄(𝑆𝑡,𝑡)]
Qtable[state][action] = Qtable[state][action] + alpha * (reward + gamma * (Qtable[next_state][next_action]) - Qtable[state][action])
```

Importantes arrays:
* `Qtable`: tabela Q com dimensão tamanho do grid ($30 \times 7=210$) por número de ações possíveis ($4$).
* `step_ep_list`: armazena o número do episódio cada vez que um novo estado é atingido. Veja que essa lista acumula o número de passos de cada episódio.
*`trajectory`: lista de estados visitados durante o episódio


In [ ]:
def gridWorld_SARSA(world, startState, goalState, alpha=0.1, gamma=1, epsilon=0.1):
    # Considere os parâmetros de entrada:
    # gama = 1 como fator de desconto
    # valores padrão de alfa e epsilon considerados, serão alterados para análise múltipla
    start_time = time.time()
    world.setTerminal(startState, goalState)
    # Inicializa Q(s,a) como q_table -> dimensão (linhas*colunas,ações possíveis)=(70,4)
    Qtable = {}
    for state in range(world.row * world.col):
        Qtable[state] = {}
        for act in world.actions_list:
            Qtable[state][act] = 0

    # função para ação epsilon-greedy
    def epsGreedy( q_dict):
        def greedyAct(_q_dict):
#            # Função retorna a ação gulosa
            greedy_act = ''
            max_q = -1e10
            for act in world.actions_list:
                if _q_dict[act] > max_q:
                    greedy_act = act
                    max_q = _q_dict[act]
            return greedy_act


        if np.random.rand() > epsilon:
           act = greedyAct(q_dict)    # ação gulosa  (exploração)
        else:
          choice = np.random.choice(world.actions_list,size = 1, p = [0.25]*4)  # escolhe ação
          act = choice[0]
        return act

    def update_Q(Qtable, state, action, reward, next_state, next_action):
       # 𝑄(𝑆𝑡,𝐴𝑡)=𝑄(𝑆𝑡,𝐴𝑡)+𝛼[𝑅𝑡+1+𝛾𝑄(𝑆𝑡+1,𝐴𝑡+1)−𝑄(𝑆𝑡,𝐴𝑡)]
       Qtable[state][action] = Qtable[state][action] + alpha * (reward + gamma * (Qtable[next_state][next_action]) - Qtable[state][action])
       return Qtable

    ep = 1                  # Número atual do episódio
    max_episodes = 300      # Número máximo de episódios
    step_ep_list = []       # lista onde serão listados os passos de cada episódio
    while ep < max_episodes:
        # Inicialização de estado 'state'
        state = world.coord_state(startState)
        trajectory = [state]
        # Escolha da ação
        action = epsGreedy(Qtable[state])
        while not world.checkTerminal(state):
            next_state = world.nextState(state, action)    # s'
            reward = world.rewardFunction(next_state)   # R(S_t+1)
            next_action = epsGreedy(Qtable[next_state])  # a'
            # atualiza tabela Q
            Qtable = update_Q(Qtable, state, action, reward, next_state, next_action)
            state = next_state
            action = next_action
            # Armazena o índice do episódio para posterior plotagem
            step_ep_list.append(ep)
            # Atualiza a trajetória do episódio
            trajectory.append(state)
        # Mostra a trajetória do agente no último episódio
        if ep == (max_episodes - 1):
            trajectoryPath(world, trajectory)
        # Aumenta o contador de episódios
        ep += 1

    plt.plot(step_ep_list, label = r'$\varepsilon$='+str(epsilon)+r', $\alpha$='+str(alpha))
    print("Tempo percorrido (em segundos): ", time.time() - start_time)
    plt.title('Windy GridWorld SARSA ', fontsize = 'large')
    plt.xlabel("Número de Steps")
    plt.ylabel("Número de Episódios")
    plt.legend()
    plt.show()


In [ ]:
startState = (3, 0)
goalState = (3, 7)
world = gridWorldWindy()
alpha = 0.5
epsilon = 0.1
print()
gridWorld_SARSA(world, startState, goalState, alpha=alpha, epsilon=epsilon)



O gráfico mostra o resultado da aplicação de $\varepsilon $-greedy Sarsa a esta tarefa, com $\varepsilon =0.1$,$\alpha = 0.5 $ e os valores iniciais $Q(s,a)=0$ para todos pares $(s,a)$. A inclinação crescente do gráfico mostra que o estado terminal é alcançado cada vez mais rapidamente ao longo do tempo.


# Q-learning


**Q-Learning** é um algoritmo de *aprendizado por reforço model free off-policy* que busca aprender a função de valor de ação ótima $Q^*(s,a)$, ou seja, o retorno esperado ao tomar a ação $a$ no estado $s$ e seguir a política ótima. O Q-learning é chamado de **off-policy** porque ele **aprende sobre a política ótima** (a que sempre escolhe a melhor ação) mesmo enquanto o agente **segue outra política durante o treinamento**, normalmente uma política **exploratória** como $\epsilon$-greedy.

Ou seja:

- A **política comportamental** (com a qual o agente gera as experiências) é exploratória.
- Mas a **política para a qual ele está aprendendo os Q-valores** é sempre a política "greedy" — que escolheria a melhor ação possível com base nos Q-valores.

Na equação de atualização:

$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ R_{t+1} + \gamma \cdot \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \right]
$$
onde,
- $s_t$ = estado atual  
- $a_t$ = ação tomada  
- $R_{t+1}$ = recompensa observada ao transitar para $s_{t+1}$  
- $\gamma$ = fator de desconto ($0 \leq \gamma < 1$)  
- $\alpha$ = taxa de aprendizado ($0 < \alpha \leq 1$)  

O termo $\max_{a'} Q(s_{t+1}, a')$ corresponde a supor que **no futuro você tomará sempre a melhor ação** — mesmo que na prática você tenha experimentado (feito uma ação aleatória) para chegar nesse $s_{t+1}$.

Por isso é "off-policy": você aprende *sobre* uma política (a ótima), mas seu comportamento segue uma política diferente (exploratória).


<center><img src='https://drive.google.com/uc?export=view&id=1FojKnSYLiGAjDcNt-1K8dstCe4iZgj0-' width="600"></center>Fonte: Richard S. Sutton and Andrew G. Barto, Reinforcement Learning: An Introduction, 2nd ed, The MIT Press.

# Sua Tarefa

<center><img src='https://drive.google.com/uc?export=view&id=156LUBNxdEIlUoj-5ayaovt-LYPKan0ml' width="400"></center>

Parta de uma política aleatória e use o método Q-learning para encontrar a política ótima. O único estad terminal é $s_8$, cuja recompensa é $+5$. Os estados $s_4,s_5$ representam perigo, com recompensa $-10$.Os demais estados são neutros. O estado inicial é o $s_0$.

In [ ]:
class GridWorld:
    def __init__(self):
        self.grid_size = 3  # Grid 3x3 (9 estados no total)
        self.state = 0      # Estado inicial (canto inferior esquerdo: S0)
        self.terminal_state = 8  # Estado terminal (canto superior direito: S8)
        self.obstacle_states = [4, 5]  # Estados com recompensa -10, nã terminais
        self.actions = ['up', 'down', 'left', 'right'] # ações possíveis

        # Mapeamento de estados para coordenadas (linha, coluna)
        self.state_coords = {i: (i // self.grid_size, i % self.grid_size)
                           for i in range(self.grid_size**2)}

    def reset(self):
        """Reseta o ambiente para o estado inicial (S0)"""
        self.state = 0
        return self.state

    def step(self, action):
        """Executa uma ação e retorna (next_state, reward, done)"""
        row, col = self.state_coords[self.state]

        # Movimentação
        if action == 'up' and row > 0:
            row -= 1
        elif action == 'down' and row < self.grid_size - 1:
            row += 1
        elif action == 'left' and col > 0:
            col -= 1
        elif action == 'right' and col < self.grid_size - 1:
            col += 1

        new_state = row * self.grid_size + col
        self.state = new_state

        # Recompensas
        if new_state == self.terminal_state:
            reward = 1
            done = True
        elif new_state in self.obstacle_states:
            reward = -10
            done = False
        else:
            reward = 0
            done = False

        return new_state, reward, done

    def render(self):
        """Visualização ASCII do grid"""
        grid = np.zeros((self.grid_size, self.grid_size), dtype=str)
        grid.fill('.')

        # Marca obstáculos (S4, S5 com X) e terminal (S8 com T)
        for s in self.obstacle_states:
            r, c = self.state_coords[s]
            grid[r, c] = 'X'

        r, c = self.state_coords[self.terminal_state]
        grid[r, c] = 'T'

        # Marca posição atual do agente com A
        r, c = self.state_coords[self.state]
        grid[r, c] = 'A'

        print(f"Estado atual: S{self.state}")
        for row in grid:
            print(' '.join(row))
        print()


In [ ]:
env = GridWorld()
state = env.reset()
env.render()

for _ in range(10):
    action = random.choice(env.actions)
    next_state, reward, done = env.step(action)
    print(f"Ação: {action:6} → Novo estado: S{next_state}, Recompensa: {reward:3}")
    env.render()

    if done:
        print("Terminal alcançado (S8)! Recompensa +1")
        break
